In [1]:
%%sql

CREATE TABLE raw_trades (
    `stream` STRING,
    `data` ROW<
        e STRING, 
        E STRING,
    	s STRING,
    	t STRING,
    	p STRING,
    	q STRING,
    	T STRING,
    	m STRING,
    	M STRING
    >,
    trade_time AS TO_TIMESTAMP_LTZ(CAST(`data`.T AS BIGINT), 3),
    WATERMARK FOR trade_time AS trade_time - INTERVAL '5' SECOND,
    proc_time AS PROCTIME()
) WITH (
    'connector' = 'kafka',
    'topic' = 'raw_trades',
    'properties.bootstrap.servers' = 'kafka:9092',
    'format' = 'json',
    'scan.startup.mode' = 'earliest-offset'
);

OK


In [ ]:
%%sql

SELECT
    data.s,
    SUM(CAST(data.p AS DECIMAL(20,8))) AS price
FROM raw_trades
    GROUP BY
        data.s

In [ ]:
%%sql

SELECT
    data.s,
    window_start,
    window_end,
    SUM(CAST(data.p AS DECIMAL(20,8))) AS price
FROM TABLE(
    HOP(TABLE raw_trades, DESCRIPTOR(trade_time), INTERVAL '5' SECONDS, INTERVAL '15' SECONDS)
)
GROUP BY
    data.s,
    window_start,
    window_end

+----+--------------------------------+-------------------------+-------------------------+------------------------------------------+
| op |                              s |            window_start |              window_end |                                    price |
+----+--------------------------------+-------------------------+-------------------------+------------------------------------------+
| +I |                        BTCUSDT | 2026-10-29 05:48:25.000 | 2026-10-29 05:48:40.000 |                        13743706.84000000 |
| +I |                        BNBUSDT | 2026-10-29 05:48:25.000 | 2026-10-29 05:48:40.000 |                           77975.27000000 |
| +I |                        XRPUSDT | 2026-10-29 05:48:25.000 | 2026-10-29 05:48:40.000 |                              24.75500000 |
| +I |                        SOLUSDT | 2026-10-29 05:48:25.000 | 2026-10-29 05:48:40.000 |                            1308.38000000 |
| +I |                        ETHUSDT | 2026-10-29 05:4